# 🇮🇳 Vani-Kanoon Legal LLM Fine-Tuning

**Fine-tune Phi-3.5-mini on Indian Legal Data for Offline Mobile Use**

This notebook will:
1. Install required packages
2. Load and prepare training data
3. Fine-tune using QLoRA (4-bit quantization)
4. Save model in HuggingFace format
5. Convert to GGUF for mobile deployment

**Requirements:**
- Kaggle GPU (T4 or P100) - FREE
- ~4 hours runtime
- Upload your `training_data.json` to Kaggle datasets

## Step 1: Install Dependencies

In [ ]:
# Install Unsloth for fast fine-tuning
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets huggingface_hub -q

In [ ]:
# Verify GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA Version: {torch.version.cuda}")

## Step 2: Configuration

In [ ]:
# ============================================
# CONFIGURATION - Edit these values
# ============================================

# Model selection (choose one)
MODEL_NAME = "unsloth/Phi-3.5-mini-instruct"  # Best for mobile (3.8B params)
# MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"  # Alternative
# MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3"  # Larger, better quality

# Training configuration
MAX_SEQ_LENGTH = 2048  # Context length
LORA_R = 32  # LoRA rank (higher = more capacity)
LORA_ALPHA = 32
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4

# Paths
TRAINING_DATA_PATH = "/kaggle/input/vani-legal-training-data/training_data.json"  # Update this
OUTPUT_DIR = "/kaggle/working/vani-legal-model"

print("Configuration loaded!")

## Step 3: Load Base Model

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading model: {MODEL_NAME}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (float16 for T4)
    load_in_4bit=True,  # Use 4-bit quantization for training
)

print("✅ Model loaded successfully!")

## Step 4: Add LoRA Adapters

In [ ]:
print("Adding LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print trainable parameters
def print_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} = {100*trainable/total:.2f}%")

print_trainable_params(model)
print("✅ LoRA adapters added!")

## Step 5: Prepare Training Data

In [ ]:
import json
from datasets import Dataset

# Prompt template for Phi-3
PHI3_TEMPLATE = """<|system|>
You are Vani, an expert AI legal assistant specializing in Indian law. You provide accurate, helpful information about Indian laws including the Bharatiya Nyaya Sanhita (BNS), Bharatiya Nagarik Suraksha Sanhita (BNSS), Constitution of India, and all other Indian statutes. Always cite relevant sections and acts. Be concise but thorough. Respond in the same language as the user's question.<|end|>
<|user|>
{instruction}{input}<|end|>
<|assistant|>
{output}<|end|>"""

# Alternative template for Llama
LLAMA_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are Vani, an expert AI legal assistant specializing in Indian law. You provide accurate, helpful information about Indian laws including the Bharatiya Nyaya Sanhita (BNS), Bharatiya Nagarik Suraksha Sanhita (BNSS), Constitution of India, and all other Indian statutes. Always cite relevant sections and acts. Be concise but thorough. Respond in the same language as the user's question.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{instruction}{input}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{output}<|eot_id|>"""

# Select template based on model
TEMPLATE = PHI3_TEMPLATE if "Phi" in MODEL_NAME else LLAMA_TEMPLATE

def format_example(example):
    """Format a single training example"""
    input_text = example.get('input', '')
    if input_text:
        input_text = f"\n\nContext: {input_text}"
    
    return TEMPLATE.format(
        instruction=example['instruction'],
        input=input_text,
        output=example['output']
    )

# Load training data
print(f"Loading training data from: {TRAINING_DATA_PATH}")

with open(TRAINING_DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} examples")

# Format and create dataset
formatted_data = [{"text": format_example(ex)} for ex in raw_data]
dataset = Dataset.from_list(formatted_data)

print(f"Dataset size: {len(dataset)}")
print(f"\nSample formatted example:\n{dataset[0]['text'][:500]}...")

## Step 6: Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting training...")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Learning rate: {LEARNING_RATE}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        warmup_steps=50,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        report_to="none",
    ),
)

# Train!
trainer_stats = trainer.train()

print(f"\n✅ Training completed!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

## Step 7: Test the Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def ask_vani(question, max_tokens=512):
    """Ask Vani a legal question"""
    if "Phi" in MODEL_NAME:
        prompt = f"""<|system|>
You are Vani, an expert AI legal assistant specializing in Indian law.<|end|>
<|user|>
{question}<|end|>
<|assistant|>
"""
    else:
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are Vani, an expert AI legal assistant specializing in Indian law.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{question}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1]
    return response.strip()

# Test questions
test_questions = [
    "What is Section 103 of BNS?",
    "How do I file an FIR in India?",
    "What are the grounds for divorce under Hindu Marriage Act?",
    "धारा 302 में क्या प्रावधान है?",  # Hindi
]

print("=" * 60)
print("TESTING FINE-TUNED MODEL")
print("=" * 60)

for q in test_questions:
    print(f"\n📝 Question: {q}")
    print(f"\n🤖 Vani: {ask_vani(q)}")
    print("-" * 60)

## Step 8: Save Model

In [ ]:
print("Saving model...")

# Save LoRA adapter only (small file)
model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print(f"✅ LoRA adapter saved to {OUTPUT_DIR}/lora")

# Merge LoRA into base model and save (for GGUF conversion)
model.save_pretrained_merged(
    f"{OUTPUT_DIR}/merged",
    tokenizer,
    save_method="merged_16bit"
)
print(f"✅ Merged model saved to {OUTPUT_DIR}/merged")

## Step 9: Convert to GGUF (For Mobile)

In [ ]:
# Install llama.cpp for GGUF conversion
!pip install llama-cpp-python -q

# Clone llama.cpp for conversion scripts
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!pip install -r /kaggle/working/llama.cpp/requirements.txt -q

In [ ]:
import subprocess
import os

MERGED_MODEL_PATH = f"{OUTPUT_DIR}/merged"
GGUF_OUTPUT_PATH = f"{OUTPUT_DIR}/vani-legal.gguf"
QUANTIZED_OUTPUT_PATH = f"{OUTPUT_DIR}/vani-legal-q4_k_m.gguf"

# Convert to GGUF (F16)
print("Converting to GGUF format...")
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_MODEL_PATH} --outfile {GGUF_OUTPUT_PATH} --outtype f16

print(f"✅ GGUF model saved to {GGUF_OUTPUT_PATH}")

In [ ]:
# Build llama.cpp quantize tool
!cd /kaggle/working/llama.cpp && make -j quantize

# Quantize to Q4_K_M (best balance of size/quality for mobile)
print("Quantizing model for mobile...")
!/kaggle/working/llama.cpp/llama-quantize {GGUF_OUTPUT_PATH} {QUANTIZED_OUTPUT_PATH} Q4_K_M

# Check file sizes
import os
f16_size = os.path.getsize(GGUF_OUTPUT_PATH) / (1024**3)
q4_size = os.path.getsize(QUANTIZED_OUTPUT_PATH) / (1024**3)

print(f"\n📊 Model Sizes:")
print(f"  F16 GGUF: {f16_size:.2f} GB")
print(f"  Q4_K_M GGUF: {q4_size:.2f} GB (for mobile)")
print(f"\n✅ Quantized model ready for mobile: {QUANTIZED_OUTPUT_PATH}")

## Step 10: Download Model

In [ ]:
# Create a zip of important files
!cd {OUTPUT_DIR} && zip -r /kaggle/working/vani-legal-model.zip \
    lora/ \
    vani-legal-q4_k_m.gguf

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"\nDownload these files:")
print(f"  1. {QUANTIZED_OUTPUT_PATH} - For mobile deployment")
print(f"  2. {OUTPUT_DIR}/lora/ - LoRA adapter (for future fine-tuning)")
print(f"\nOr download the zip: /kaggle/working/vani-legal-model.zip")

## Optional: Upload to Hugging Face

In [ ]:
# Uncomment and run this cell to upload to Hugging Face

# from huggingface_hub import HfApi, login
# 
# # Login (get token from huggingface.co/settings/tokens)
# login(token="YOUR_HF_TOKEN_HERE")
# 
# # Upload
# api = HfApi()
# api.upload_folder(
#     folder_path=f"{OUTPUT_DIR}/merged",
#     repo_id="YOUR_USERNAME/vani-legal-phi3",
#     repo_type="model",
# )
# 
# # Upload GGUF
# api.upload_file(
#     path_or_fileobj=QUANTIZED_OUTPUT_PATH,
#     path_in_repo="vani-legal-q4_k_m.gguf",
#     repo_id="YOUR_USERNAME/vani-legal-phi3-gguf",
#     repo_type="model",
# )

---

## 📱 Next Steps: Mobile Integration

1. Download `vani-legal-q4_k_m.gguf` (~2.3 GB)
2. Add to your Capacitor Android app under `android/app/src/main/assets/models/`
3. Use `capacitor-llama` plugin to load and run the model

See the mobile integration code in your project's `frontend/src/services/offlineLegalLLM.js`